In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

# ---------------------------------------------------------------
# Source resolution with fail-fast guards
# ---------------------------------------------------------------
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
splits = json.loads((ROOT / "splits" / "split_indices.json").read_text())
curated = pd.read_csv(ROOT / splits["curated_csv"]).reset_index(drop=True)

expected_total = sum(len(splits["indices"][s]) for s in ["train", "validation", "test"])
if len(curated) != expected_total:
    raise RuntimeError(
        f"Loaded {len(curated)} rows from {splits['curated_csv']}, but "
        f"split_indices expects {expected_total}. Wrong source file — aborting "
        f"before writing stale ml_ad_scores.csv."
    )
for s in ["train", "validation", "test"]:
    classes = curated.iloc[splits["indices"][s]]["activity_class"].unique()
    if "ambiguous" in classes:
        raise RuntimeError(
            f"{s} split contains ambiguous records — loaded {splits['curated_csv']} "
            f"but split expects eligible-only data."
        )

print(f"Source: {splits['curated_csv']} ({len(curated)} rows)")

# ---------------------------------------------------------------
# Load model and rosemary catalogue
# ---------------------------------------------------------------
model = joblib.load(ROOT / "models" / "selected_model_rf.pkl")
rosemary = pd.read_csv(ROOT / "data" / "cultivar_compound_catalogue.csv").drop_duplicates("compound_id")

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# ---------------------------------------------------------------
# 1. Generate features for Rosemary
# ---------------------------------------------------------------
def get_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return morgan_gen.GetFingerprint(mol) if mol else None

rosemary_fps = [get_fp(s) for s in rosemary["canonical_smiles"]]
X_rosemary = np.array([list(fp) for fp in rosemary_fps])

# ---------------------------------------------------------------
# 2. Predict ML scores
# ---------------------------------------------------------------
raw_probs = model.predict_proba(X_rosemary)[:, 1]

# Reference distribution: validation-set predictions (correct 30-compound set now)
val_smiles = curated.iloc[splits["indices"]["validation"]]["canonical_smiles"]
X_val = np.array([list(get_fp(s)) for s in val_smiles])
val_probs = model.predict_proba(X_val)[:, 1]

# ML_rank_score = fraction of validation compounds at or below this compound's probability
ml_rank_scores = [(val_probs <= p).sum() / len(val_probs) for p in raw_probs]
rosemary["ML_raw_score"] = raw_probs
rosemary["ML_rank_score"] = ml_rank_scores

# ---------------------------------------------------------------
# 3. Applicability Domain (AD)
# ---------------------------------------------------------------
train_smiles = curated.iloc[splits["indices"]["train"]]["canonical_smiles"]
train_fps = [get_fp(s) for s in train_smiles]

# Threshold: 95th percentile of training LOO mean Jaccard distance (k=5)
train_distances = []
for i, fp in enumerate(train_fps):
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    sims.pop(i)  # remove self-similarity
    sims.sort(reverse=True)
    train_distances.append(np.mean([1 - s for s in sims[:5]]))
ad_threshold = float(np.percentile(train_distances, 95))
print(f"AD threshold: {ad_threshold:.4f}")

# Rosemary AD distances
ad_results = []
for fp in rosemary_fps:
    sims = DataStructs.BulkTanimotoSimilarity(fp, train_fps)
    sims.sort(reverse=True)
    dist = float(np.mean([1 - s for s in sims[:5]]))
    ad_results.append(dist)

rosemary["ad_distance"] = ad_results
rosemary["ad_flag"] = np.where(
    rosemary["ad_distance"] <= ad_threshold, "within_AD", "extrapolation"
)

# ---------------------------------------------------------------
# 4. Report
# ---------------------------------------------------------------
print(f"\nAD Summary:")
print(rosemary["ad_flag"].value_counts().to_string())
print(f"\nML_rank_score diagnostics:")
print(f"  unique values: {rosemary['ML_rank_score'].nunique()}")
print(f"  top band: {rosemary['ML_rank_score'].value_counts().head(3).to_dict()}")

# ---------------------------------------------------------------
# 5. Save
# ---------------------------------------------------------------
out = rosemary[[
    "compound_id",
    "ML_raw_score",
    "ML_rank_score",
    "ad_distance",
    "ad_flag",
]]
out_path = ROOT / "results" / "ml_ad_scores.csv"
out.to_csv(out_path, index=False)
print(f"\nSaved {len(out)} rows to {out_path}")

# ---------------------------------------------------------------
# 6. Cross-check against ad_results.csv (from 02) — same math, same training set
# ---------------------------------------------------------------
ad_02_path = ROOT / "results" / "ad_results.csv"
if ad_02_path.exists():
    ad_02 = pd.read_csv(ad_02_path)
    merged = out.merge(
        ad_02[["compound_id", "ad_distance"]],
        on="compound_id",
        suffixes=("_03", "_02"),
    )
    merged["diff"] = (merged["ad_distance_03"] - merged["ad_distance_02"]).abs()
    max_diff = float(merged["diff"].max())
    if max_diff > 1e-4:
        print(f"\nWARNING: AD distance differs from ad_results.csv by up to {max_diff:.6f}")
        print("  03 and 02 may compute AD differently — investigate before using either.")
    else:
        print(f"\nAD consistency check passed (max diff vs ad_results.csv = {max_diff:.2e})")

Source: data/curated_chembl_eligible.csv (179 rows)
AD threshold: 0.7330

AD Summary:
ad_flag
extrapolation    28

ML_rank_score diagnostics:
  unique values: 2
  top band: {0.6333333333333333: 27, 0.6: 1}

Saved 28 rows to c:\Users\aymen\Downloads\Romarinova-AI\results\ml_ad_scores.csv

AD consistency check passed (max diff vs ad_results.csv = 1.11e-16)
